# Lara's notities

## Preprocessen/sample methoden
The pipeline described by MNE is https://mne.tools/stable/documentation/cookbook.html is VERY detailed. I think our data is already quite polished maybe? They only talk about downsampling and normalization in the project description so maybe such things are more complicated than needed. 
Do downsampling with basic factor 10 right now and do average pooling, factor can be bigger when slow. PCA for dimension reduction?
Z-score normalization can be better then min-max when amplitudes vary a lot for the MEG data (z-score uses the normal distribution while min-max is linear, https://www.codecademy.com/article/min-max-zscore-normalization).

Total steps: read data, downsample data, stack the data to one x_train, normalize the data.

## (a) Modellen om uit te kiezen
MNE tools are made for MEG analysis and visualization. Also learning? sklearn. Could do normal classification too maybe. 

I chose 1D CNN becauseee it seems fine (survey: https://www.sciencedirect.com/science/article/pii/S0888327020307846).

Model: 1D CNN, optimizer Adam, Loss function Entropyloss. Memory management with batches of size 16.
No grid search on parameters or other models.

## (b) Test framework voor zowel cross als intra testen
Done.

## (c) Hyperparameter tweaking framework
TO DO!

We moeten vooral ook beschrijven hoe we verbeteren enzo. Miss een soort labjournaal bijhouden?

In [67]:
%pip install torch numpy scipy scikit-learn matplotlib h5py

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Read data

In [ ]:
import os
import glob
import h5py
import numpy as np

# Functions to load/get/extract adminstrative stuff
def get_dataset_name(file_name_with_dir):
    file_name_without_dir = os.path.basename(file_name_with_dir)
    temp = file_name_without_dir.split(".")[:-1]
    dataset_name = "".join(temp)
    return dataset_name

def load_h5_file(filepath):
    with h5py.File(filepath, 'r') as f: # use with to close it interestingly
        dataset_name = list(f.keys())[0]
        return f[dataset_name][()]      # Outputs matrix that is the numpy array of the data in the h5py file

def get_label(filepath):    # extract classification activity label: 0="rest", 1="math", 2="memory" en 3="motor"
    name = os.path.basename(filepath)
    if name.startswith("rest"):     # maybe slow sorry, idk how else to do this. 10s valt mee
        return 0
    elif "math" in name:
        return 1
    elif "memory" in name:
        return 2
    elif "motor" in name:
        return 3
    else:
        raise ValueError(f"Unknown label for file: {name}")     # This happens apparently :/ OHWWHW it starts with task

# downsample when loading data
def downsample(matrix, factor=10):
    nsensors, ntimesteps = matrix.shape
    down_ntimesteps = ntimesteps // factor
    matrix = matrix[:, :down_ntimesteps * factor]
    matrix = matrix.reshape(nsensors,down_ntimesteps,factor).mean(axis=2)
    return matrix

# load a folder of data!
def load_folder(folder):
    x = []
    y = []
    files = glob.glob(os.path.join("data", folder, "*.h5"))
    # print(files)
    for file in files:
        matrix = load_h5_file(file)
        matrix = downsample(matrix)
        x.append(matrix)        # now has for every file a matrix with different measurers and different timesteps, so (number, channels, timesteps)
        y.append(get_label(file))
    return np.array(x), np.array(y)

# Load all the data (maybe requires too much memory?)
x_train_intra, y_train_intra = load_folder(r"Intra\train")
x_test_intra, y_test_intra = load_folder(r"Intra\test")

x_train_cross, y_train_cross = load_folder(r"Cross\train")
x_test_cross1, y_test_cross1 = load_folder(r"Cross\test1")
x_test_cross2, y_test_cross2 = load_folder(r"Cross\test2")
x_test_cross3, y_test_cross3 = load_folder(r"Cross\test3")

# Preprocessing
z-score normalization

In [69]:
from sklearn.preprocessing import StandardScaler

# Normalization via z-score sklearn standardscaler. We have to scale back the data to a 2d array first, then we normalize, then we construct the train/test again.
def normalize(train, test):
    n, c, t = train.shape
    scaler = StandardScaler()
    train_2d = train.transpose(0, 2, 1).reshape(-1, c)
    test_2d  = test.transpose(0, 2, 1).reshape(-1, c)
    train_2d = scaler.fit_transform(train_2d)
    test_2d = scaler.transform(test_2d)
    train = train_2d.reshape(n, t, c).transpose(0, 2, 1)
    test  = test_2d.reshape(test.shape[0], t, c).transpose(0, 2, 1)
    return train, test

# Normalize all data (repeated code oh well)
x_train_intra, x_test_intra = normalize(x_train_intra, x_test_intra)
x_train_cross, x_test_cross1 = normalize(x_train_cross, x_test_cross1)
_, x_test_cross2 = normalize(x_train_cross, x_test_cross2)
_, x_test_cross3 = normalize(x_train_cross, x_test_cross3)

# Model training and testing methods

In [70]:
import torch
from torch.utils.data import Dataset
import torch.nn as nn

from torch.utils.data import DataLoader
import torch.optim as optim

# We use torch once again. This modifies data to make it fit.
class Torchdata(Dataset):
    def __init__(self,x,y):
        x = np.asarray(x, dtype=np.float32)
        y = np.asarray(y, dtype=np.int64)
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)  # torch has longs, python 3.x doenst
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self,xid):
        return self.x[xid], self.y[xid]

# we use the 1d convolutional nn. Did hardcoding first of sizes, is now dynamic.
class CNN(nn.Module):
    def __init__(self, channels, input_lenght):
        super().__init__()

        self.conv1 = nn.Conv1d(channels, 64, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.pool = nn.MaxPool1d(4)
        self.relu = nn.ReLU()

        with torch.no_grad():
            dummy = torch.zeros(1, channels, input_lenght)
            x = self.pool(self.relu(self.conv1(dummy)))
            x = self.pool(self.relu(self.conv2(x)))
            self.dyn_size = x.shape[1] * x.shape[2]
        self.fc1 = nn.Linear(self.dyn_size, 128)
        self.fc2 = nn.Linear(128, 4)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# Trainng method. Epochs can change ofc. Not yet cross validation on parameters
def train_model(x_train,y_train,epochs=100):
    dataset = Torchdata(x_train,y_train)
    loader = DataLoader(dataset, batch_size=16, shuffle=True)       # memory management!
    model = CNN(x_train.shape[1],x_train.shape[2])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = optim.Adam(model.parameters(),lr=0.001)
    loss_function = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = loss_function(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print("Epoch", epoch, "Loss:", total_loss)
        if total_loss == 0: break       # Yes, this happens
    return model
    
# Evaluation, returns accuracy
def evaluate(model, x, y):
    dataset = Torchdata(x, y)
    loader = DataLoader(dataset, batch_size=16)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model(x_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    return correct / total

# Do the training and the testing
Na een aantal epochs als de loss functie duidelijk geconvergeerd is naar 0 gaat hij nog steeds door :/. Nu gefixt met break.

## Result
Took me only 2 m 5.8 s
Intra-subject accuracy: 1.0
Cross-subject 1 accuracy: 0.6875
Cross-subject 2 accuracy: 0.25
Cross-subject 3 accuracy: 0.25

Without downsampling:
Took me 2 m 51.7 s
Intra-subject accuracy: 1.0
Cross-subject 1 accuracy: 0.8125
Cross-subject 2 accuracy: 0.25
Cross-subject 3 accuracy: 0.25

In [71]:
# Intra
model_intra = train_model(x_train_intra, y_train_intra)
acc_intra = evaluate(model_intra, x_test_intra, y_test_intra)
print("Intra-subject accuracy:", acc_intra)

# Cross
model_cross = train_model(x_train_cross, y_train_cross)
acc_cross1 = evaluate(model_cross, x_test_cross1, y_test_cross1)
acc_cross2 = evaluate(model_cross, x_test_cross2, y_test_cross2)
acc_cross3 = evaluate(model_cross, x_test_cross3, y_test_cross3)
print("Cross-subject 1 accuracy:", acc_cross1)
print("Cross-subject 2 accuracy:", acc_cross2)
print("Cross-subject 3 accuracy:", acc_cross3)

Epoch 0 Loss: 29.80665421485901
Epoch 1 Loss: 14.532795429229736
Epoch 2 Loss: 2.738509237766266
Epoch 3 Loss: 0.0002553842455768063
Epoch 4 Loss: 3.725281771949085e-07
Epoch 5 Loss: 4.470332441997016e-07
Epoch 6 Loss: 1.4901159417490817e-08
Epoch 7 Loss: 0.0
Intra-subject accuracy: 1.0
Epoch 0 Loss: 49.82254230976105
Epoch 1 Loss: 2.47680444130674
Epoch 2 Loss: 2.4759186923038214
Epoch 3 Loss: 5.1502845138311315
Epoch 4 Loss: 0.21689526979561435
Epoch 5 Loss: 2.0217094623149023
Epoch 6 Loss: 1.798741027712225
Epoch 7 Loss: 4.3091544806957245
Epoch 8 Loss: 0.7472085356712341
Epoch 9 Loss: 4.4703469370688254e-08
Epoch 10 Loss: 2.5933876037597656
Epoch 11 Loss: 0.0
Cross-subject 1 accuracy: 0.8125
Cross-subject 2 accuracy: 0.25
Cross-subject 3 accuracy: 0.25
